In [ ]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
import requests
from langchain_community.tools import DuckDuckGoSearchRun
from dotenv import load_dotenv

load_dotenv()

search_tool = DuckDuckGoSearchRun() ## tool for an internet search

@tool
def get_weather_data(city: str) -> str: ## tool to get a weather data
    """Fetch current weather data of a given city"""

    # Step 1: Convert city → latitude & longitude
    geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}"
    geo_res = requests.get(geo_url).json()

    if "results" not in geo_res:
        return f"Could not find location for {city}"

    lat = geo_res["results"][0]["latitude"]
    lon = geo_res["results"][0]["longitude"]

    # Step 2: Get weather
    weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
    weather_res = requests.get(weather_url).json()

    return str(weather_res["current_weather"])

llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)

from langchain_classic.agents.react.agent import create_react_agent
from langchain_classic.agents.agent import AgentExecutor
from langchain_classic import hub

prompt = hub.pull("hwchase17/react") ## Pull react prompt agent
# This prompt teaches the LLM:
# How to think step-by-step
# When to use tools
# Output format

agent = create_react_agent(llm=llm , tools=[search_tool, get_weather_data] , prompt=prompt)
#What is a ReAct Agent?

#ReAct = Reason + Act

#It follows a loop like: thought , action observation
## Then give final answer

agent_executor = AgentExecutor(agent=agent, tools=[search_tool, get_weather_data] , verbose=True)
# This actually:
# Runs the loop
# Calls tools
# Passes results back to LLM
# Stops when final answer is ready

response = agent_executor.invoke({'input':"find capital of Madhya Pradesh and its weather"})

print(response['output'])

## LLM (ChatGroq) → Brain (decides what to do)
## Agent (ReAct) → Strategy (how to think step-by-step)
## AgentExecutor → Operator (actually executes tools)
## Tools → Workers (do real tasks like API calls)



> Entering new AgentExecutor chain...
Thought: To find the capital of Madhya Pradesh, I should first look up the information about the state.
Action: duckduckgo_search
Action Input: capital of Madhya PradeshIndore (/ ɪnˈdɔːr / ⓘ; ISO: Indaura, Hindi: [ɪn̪d̪ɔːr]) is the largest city of the Indian state ofMadhyaPradeshwhere it serves as thecapitaland the administrative headquarters of the eponymous district and division. Modern-day Indore was established on the banks of the Kanh and Saraswati rivers and traces its roots to its 16th-century founding as a trading hub between the ... MadhyaPradesh, state of India that is situated in the heart of the country. It has no coastline and no international frontier. Its physiography is characterized by low hills, extensive plateaus, and river valleys. Thecapitalis Bhopal, in the west-central part of the state. In November 2000, India gained three new states - Chattisgarh carved out ofMadhyaPradesh, Uttaranchal from UttarPradesh, and Jharkhand fro